# 18 · PPO — learning *online* from the execution reward

**Recap (16–17):** DPO trained on a **fixed** set of preference pairs (offline). PPO is
the **online** alternative: the model **generates** a response, gets it **scored**, and
**updates** — over and over, on fresh samples. With the execution reward standing in for
a learned reward model, this is RLHF-style training. It's the last piece of the
objective axis.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

## Step 1 — The online loop

PPO repeats three steps:

1. **Sample** a response from the current policy.
2. **Score** it with the reward (here: `execution_reward` — run the SQL, compare to gold).
3. **Update** the policy toward responses that scored well.

Unlike DPO, the training data is *generated fresh each step* by the model itself
(*on-policy*). That's the power (it can explore) and the pain (it's noisier).

## Step 2 — The core idea: policy gradient ("reinforce what worked")

The rule is intuitive: **if a sampled response earned a high reward, make the tokens that
produced it more likely; if low, less likely.** Here's a toy with 3 candidate SQLs and
their execution rewards — the policy starts uniform and learns by sampling:

In [ ]:
rewards = np.array([0.3, 1.0, 0.0])     # candidate 1 = correct (reward 1.0)
logits  = np.zeros(3)                    # policy starts uniform over the 3 candidates

lr = 0.5
for _ in range(300):
    p = softmax(logits)
    a = rng.choice(3, p=p)               # SAMPLE a candidate (the 'online' part)
    grad = -p.copy(); grad[a] += 1       # d log p(a) / d logits
    logits += lr * rewards[a] * grad     # reinforce in proportion to the reward

print("final policy:", softmax(logits).round(3))
print("-> mass concentrates on candidate 1, the reward-1.0 correct query")

## Step 3 — The danger: pure reward-chasing collapses (and can be *hacked*)

Notice the policy went almost entirely to `[0, 1, 0]` — it **abandoned** candidate 0
(the `0.3` "valid but wrong" query) completely, and would happily collapse onto *any*
quirk that scores high, including ways to **game** an imperfect reward (*reward hacking*).
Unconstrained reward maximization is brittle.

## Step 4 — The fix: a KL leash to the reference

PPO adds a penalty for drifting too far from a frozen **reference** policy (the SFT
model) — measured by **KL divergence**. Intuition: *"chase reward, but don't wander far
from the sensible model you started with."* This keeps outputs fluent and blocks the
worst reward-hacking. A coefficient (often called `β` or the KL coefficient) sets how
tight the leash is — exactly the same *leash* idea you met in DPO (notebook 16), just
applied during an online loop instead of baked into the loss.

## Step 5 — The "Proximal" part (in one line)

PPO also **clips** how much the policy can change in a single update, so one noisy batch
can't lurch it somewhere bad. That clipped, small-step update is what the *Proximal* in
**P**roximal **P**olicy **O**ptimization refers to — it's the trick that makes online RL
stable enough to use.

## Step 6 — In this lab

`preference.train_ppo` does exactly Step 1–5: it samples a SQL from the policy, scores it
with `eval_harness.execution_reward` (no learned reward model — the **SQL engine is the
judge**), and updates via TRL's `PPOTrainer`. As [CLAUDE.md](CLAUDE.md) warns, PPO is the
finickiest path and needs a real GPU run to validate. The reward is **execution truth**,
and the one classic-RLHF fork would be swapping a learned reward model in here.

## Recap — the objective axis is complete

| objective | data | reward | online? |
|---|---|---|---|
| **SFT** | one gold answer | (imitation) | offline |
| **DPO** | fixed (chosen, rejected) pairs | implicit (`β·log π/π_ref`) | offline |
| **PPO** | freshly sampled responses | execution reward, live | **online** |

Same model, same PEFT mechanism, same execution-accuracy eval — only the **loss + data
shape** change. That's the objective axis, and you've now covered all of it.

**Triple BAM!** The whole curriculum — mechanism axis (01–12) and objective axis
(13–18) — is done. The only thing left is the *outcome* evidence: real GPU runs on
Kaggle, scoring every method through the one shared execution-accuracy harness.